In [6]:
# Step 1: Pre Set-Up

import torch
import torchvision
import torchvision.transforms as transforms

# Set up image Size
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 2. Downloading the actual road sign images (GTSRB dataset)
print("Downloading In Phase dashcam dataset")
train_data = torchvision.datasets.GTSRB(root='./data', split='train', download=True, transform=transform)

print(f"Downloaded {len(train_data)} training images.")

Downloaded 26640 training images.


In [7]:
# Step 2: Simulating varied driving environments by editing dataset
dashcam_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    # Rotation to simulate camera vibration/mounting angles that may happen on dashcams
    transforms.RandomRotation(15),

    # Lighting transforms for night driving and solar glare
    transforms.ColorJitter(brightness=0.5, contrast=0.5),

    # Blur to simulate motion/rain on the lens
    transforms.GaussianBlur(kernel_size=3),

    transforms.ToTensor()
])

# Initialize the training dataset with dashcam transforms
train_data = torchvision.datasets.GTSRB(
    root='./data',
    split='train',
    download=True,
    transform=dashcam_transform
)
# Data augmentation to simulate varied driving environments
dashcam_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    # Rotation to simulate camera vibration or mounting angles
    transforms.RandomRotation(15),

    # Lighting transforms for night driving and solar glare
    transforms.ColorJitter(brightness=0.5, contrast=0.5),

    # Blur to simulate motion or rain on the lens
    transforms.GaussianBlur(kernel_size=3),

    transforms.ToTensor()
])

# Initialize the training dataset with dashcam-specific transforms
train_data = torchvision.datasets.GTSRB(
    root='./data',
    split='train',
    download=True,
    transform=dashcam_transform
)

print("Dashcam edits applied.")

Dashcam edits applied.


In [ ]:
# Step 3: Making the Model
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Setuping data loader (just normal syntax)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)

# Using MobileNetV3_Small to be more efficient on dashcam hardware
print("Loading pre-trained MobileNetV3 weights")
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

# Fine-tuning by adjusting the final layer for all the road sign classes
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, 43)

# Defining loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Main training loop
model.train()
print("started Training")

for i, (images, labels) in enumerate(train_loader):
    optimizer.zero_grad()

    # Forward pass and loss calculation
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass and weight update
    loss.backward()
    optimizer.step()

    if i % 50 == 0:
        print(f"Batch {i} | Loss: {loss.item():.4f}")

    # Limit to 200 batches as at the end of the day its a proof of concept
    if i == 200:
        break

print("\nTraining complete.")

Loading pre-trained MobileNetV3 weights
started Training
Batch 0 | Loss: 3.8034


In [ ]:
!pip install onnx onnxscript

In [ ]:
# Final Step: Exporting to ONNX!
dummy_input = torch.randn(1, 3, 224, 224)

# Set to eval mode to freeze weights
model.eval()

onnx_file_path = "road_angel_dashcam_model.onnx"
print(f"Exporting model to {onnx_file_path}...")

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],

    # Allow for variable batch sizes during inference
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Success hopefully: {onnx_file_path} generated.")